# DTL-5G Code Sample 2 — 5G-NIDD BiLSTM Baseline
## Objective 1, iteration 1 (`random_state = 42`)

This notebook trains the BiLSTM from scratch on the 5G-NIDD target dataset,
providing the target-domain baseline without transfer learning.

Five Objective-1 iterations use fixed random states:
`42, 142, 12, 4, 80`.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Activation, Bidirectional, Dense, Dropout, LSTM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

OBJECTIVE1_RANDOM_STATES = [42, 142, 12, 4, 80]
ITERATION_SEED = OBJECTIVE1_RANDOM_STATES[0]  # Objective 1, iteration 1

DATA_ROOT = Path(os.getenv("NIDD_DATA_ROOT", "./data/5G-NIDD")).expanduser()
ATTACK_DIR = DATA_ROOT / "Attack"
BENIGN_DIR = DATA_ROOT / "Benign"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
# Reproducible model initialization, dropout masks, NumPy operations,
# and TensorFlow random operations for this iteration.
keras.utils.set_random_seed(ITERATION_SEED)

# Request deterministic TensorFlow kernels where supported.
try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print(f"Deterministic TensorFlow operations unavailable: {exc}")

## 2. Features and 5G-NIDD data manifest

In [ ]:
FEATURES = ['Flow Duration', 'Fwd Pkt Len Std', 'ACK Flag Cnt', 'Protocol', 'Tot Fwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Tot Bwd Pkts']
LABEL_COLUMN = "Label"

ATTACK_FILES = ['SYNflood_BS1_nogtp.pcap_Flow.csv', 'SYNflood_BS2_nogtp.pcap_Flow.csv', 'UDPflood_BS1_nogtp.pcap_Flow.csv', 'UDPflood_BS2_nogtp.pcap_Flow.csv', 'Goldeneye_BS1_nogtp.pcapng_Flow.csv', 'Goldeneye_BS2_nogtp.pcapng_Flow.csv', 'ICMPflood_BS1_nogtp.pcapng_Flow.csv', 'ICMPflood_BS2_nogtp.pcapng_Flow.csv', 'Slowloris_BS1_nogtp.pcapng_Flow.csv', 'Slowloris_BS2_nogtp.pcapng_Flow.csv', 'UDPScan_BS2_nogtp.pcapng_Flow.csv', 'UDPScan_BS1_nogtp.pcapng_Flow.csv', 'SYNscan_BS2_nogtp.pcapng_Flow.csv', 'SYNScan_BS1_nogtp.pcapng_Flow.csv', 'TCPConnect_BS1_nogtp.pcapng_Flow.csv', 'TCPConnect_BS2_nogtp.pcapng_Flow.csv', 'Torshammer_BS1_nogtp.pcapng_Flow.csv', 'Torshammer_BS2_nogtp.pcapng_Flow.csv']
BENIGN_FILES = ['Goldeneye_BS1_nogtp.pcap_Flow.csv', 'Goldeneye_BS2_nogtp.pcap_Flow.csv', 'ICMPflood_BS1_nogtp.pcap_Flow.csv', 'ICMPflood_BS2_nogtp.pcap_Flow.csv', 'Slowloris_BS1_nogtp.pcap_Flow.csv', 'Slowloris_BS2_nogtp.pcap_Flow.csv', 'SYNflood_BS1_nogtp.pcap_Flow.csv', 'SYNflood_BS2_nogtp.pcap_Flow.csv', 'SYNScan_BS1_nogtp.pcap_Flow.csv', 'SYNscan_BS2_nogtp.pcap_Flow.csv', 'UDPflood_BS1_nogtp.pcap_Flow.csv', 'UDPflood_BS2_nogtp.pcap_Flow.csv', 'SSH_BS1_nogtp.pcap_Flow.csv', 'SSH_BS2_nogtp.pcap_Flow.csv', 'TCPConnect_BS1_nogtp.pcap_Flow.csv', 'TCPConnect_BS2_nogtp.pcap_Flow.csv', 'Torshammer_BS1_nogtp.pcap_Flow.csv', 'Torshammer_BS2_nogtp.pcap_Flow.csv', 'UDPScan_BS1_nogtp.pcap_Flow.csv', 'UDPScan_BS2_nogtp.pcap_Flow.csv']

EXPECTED_BENIGN_COUNT = 15_212
EXPECTED_ATTACK_COUNT = 315_665

def assert_unique_manifest(name, filenames):
    duplicates = sorted({f for f in filenames if filenames.count(f) > 1})
    if duplicates:
        raise ValueError(f"{name} contains duplicate filenames: {duplicates}")

assert_unique_manifest("BENIGN_FILES", BENIGN_FILES)
assert_unique_manifest("ATTACK_FILES", ATTACK_FILES)


## 3. Load the target domain

In [ ]:
def load_group(directory, filenames, label):
    paths = [directory / name for name in filenames]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing 5G-NIDD CSV files. Set NIDD_DATA_ROOT.\n"
            + "\n".join(missing[:10])
        )

    frame = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
    frame[LABEL_COLUMN] = label
    return frame[FEATURES + [LABEL_COLUMN]]

attack_df = load_group(ATTACK_DIR, ATTACK_FILES, "attack")
benign_df = load_group(BENIGN_DIR, BENIGN_FILES, "Benign")

print("Benign:", f"{len(benign_df):,}")
print("Attack:", f"{len(attack_df):,}")

assert len(benign_df) == EXPECTED_BENIGN_COUNT
assert len(attack_df) == EXPECTED_ATTACK_COUNT

## 4. Cleaning and fixed Objective-1 balancing

In [ ]:
attack_df = attack_df.replace([np.inf, -np.inf], np.nan)
benign_df = benign_df.replace([np.inf, -np.inf], np.nan)

def validate_selected_features(frame, name):
    values = frame[FEATURES].to_numpy(dtype=np.float64)
    invalid = ~np.isfinite(values)

    if invalid.any():
        rows, cols = np.where(invalid)
        examples = [
            (int(r), FEATURES[int(c)])
            for r, c in zip(rows[:10], cols[:10])
        ]
        raise ValueError(
            f"{name} contains {invalid.sum()} non-finite selected-feature values. "
            f"Examples: {examples}"
        )

    print(f"{name}: selected features are finite.")

validate_selected_features(attack_df, "5G-NIDD attack traffic")
validate_selected_features(benign_df, "5G-NIDD benign traffic")


attack_df = attack_df.sample(
    n=len(benign_df),
    random_state=ITERATION_SEED,
    replace=False,
)

print("Balanced benign:", len(benign_df))
print("Balanced attack:", len(attack_df))

## 5. Feature matrix and labels

In [ ]:
X_benign = benign_df[FEATURES].to_numpy(dtype=np.float32)
X_attack = attack_df[FEATURES].to_numpy(dtype=np.float32)

X = np.concatenate((X_benign, X_attack), axis=0)
y = np.concatenate((
    np.ones(len(benign_df), dtype=np.int32),
    np.zeros(len(attack_df), dtype=np.int32),
))

assert X.shape == (30_424, 8)
print("X:", X.shape)
print("y:", y.shape)

## 6. Stratified 80/20 split and StandardScaler

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=ITERATION_SEED,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

assert len(X_test) == 6_085
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 7. BiLSTM baseline trained from scratch

In [ ]:
def build_bilstm(input_shape=(8, 1), num_classes=2):
    """
    BiLSTM graph satisfying the explicit experiment specifications:
      - 9 top-level Keras layers
      - 99,970 parameters
      - L2 coefficient 0.001
      - dropout rate 0.5
      - Adam learning rate 1e-5

    Dense(256) + explicit ReLU is mathematically equivalent to
    Dense(256, activation="relu") but keeps the reported layer count explicit.
    """
    reg = regularizers.l2(0.001)

    inputs = keras.Input(shape=input_shape, name="input")
    x = Bidirectional(
        LSTM(
            64,
            activation="tanh",
            kernel_regularizer=reg,
        ),
        name="bilstm",
    )(inputs)
    x = Dropout(0.5, name="dropout_1")(x)
    x = Dense(
        256,
        activation=None,
        kernel_regularizer=reg,
        name="dense_256",
    )(x)
    x = Activation("relu", name="relu_256")(x)
    x = Dropout(0.5, name="dropout_2")(x)
    x = Dense(
        128,
        activation="relu",
        kernel_regularizer=reg,
        name="dense_128",
    )(x)
    x = Dropout(0.5, name="dropout_3")(x)
    outputs = Dense(
        num_classes,
        activation="sigmoid",
        kernel_regularizer=reg,
        name="output",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="DTL5G_BiLSTM")
    model.compile(
        loss="binary_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        metrics=["accuracy"],
    )

    assert len(model.layers) == 9, f"Expected 9 layers, found {len(model.layers)}"
    assert model.count_params() == 99_970, (
        f"Expected 99,970 parameters, found {model.count_params():,}"
    )
    return model

model = build_bilstm()
model.summary()

print("Top-level layers:", len(model.layers))
print("Total parameters:", f"{model.count_params():,}")

## 8. Train

In [ ]:
y_train_one_hot = keras.utils.to_categorical(y_train.astype(float), num_classes=2)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    min_delta=0.0001,
)

start = time.perf_counter()
history = model.fit(
    X_train,
    y_train_one_hot,
    epochs=200,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1,
)
training_seconds = time.perf_counter() - start

print(f"Training time: {training_seconds / 60:.2f} min")
print(f"Training time/sample: {(training_seconds * 1000) / len(X_train):.4f} ms")

## 9. Save target baseline

In [ ]:
TARGET_BASELINE_PATH = MODEL_DIR / "BiLSTM_5GNIDD_Obj1_1.keras"
model.save(TARGET_BASELINE_PATH)
print("Saved:", TARGET_BASELINE_PATH)

## 10. Evaluate

In [ ]:
def evaluate_attack_positive(model, X_test, y_test):
    """Evaluate using the evaluation convention: attack is the positive class."""
    probabilities = model.predict(X_test, verbose=0)
    predictions = probabilities.argmax(axis=1)

    metrics_out = {
        "accuracy": accuracy_score(y_test, predictions),
        "recall": recall_score(y_test, predictions, pos_label=0, zero_division=0),
        "precision": precision_score(y_test, predictions, pos_label=0, zero_division=0),
        "f1": f1_score(y_test, predictions, pos_label=0, zero_division=0),
        "auc": roc_auc_score((y_test == 0).astype(int), probabilities[:, 0]),
    }

    for key, value in metrics_out.items():
        print(f"{key.capitalize():>9}: {value:.4f}")

    return probabilities, predictions, metrics_out

probabilities, predictions, result = evaluate_attack_positive(
    model, X_test, y_test
)

## 11. Inference timing

In [ ]:
def benchmark(device):
    with tf.device(device):
        start = time.perf_counter()
        _ = model.predict(X_test, verbose=0)
        elapsed = time.perf_counter() - start
    return elapsed, elapsed * 1000 / len(X_test)

_, cpu_each = benchmark("/CPU:0")
print(f"CPU: {cpu_each:.4f} ms/sample")

if tf.config.list_physical_devices("GPU"):
    _, gpu_each = benchmark("/GPU:0")
    print(f"GPU: {gpu_each:.4f} ms/sample")

## 12. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Attack", "Benign"],
    columns=["Attack", "Benign"],
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cbar=False)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("5G-NIDD BiLSTM baseline — Objective 1, iteration 1")
plt.tight_layout()
plt.show()

## Five-run aggregation

This notebook contains one Objective-1 iteration. The complete experiment uses
the fixed seeds `42, 142, 12, 4, 80`, and final metrics are obtained by averaging
across the five runs.
